# Epochs — Model Training & Cross-Validation

Compare Logistic Regression, KNN, SVM, and Random Forest using artist-grouped stratified CV and GridSearchCV.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

PROJECT_ROOT = Path.cwd().parent
df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "fma_features.csv")

ID_COLUMNS = ["track_id","artist_id","genre"]
feature_columns = [c for c in df.columns if c not in ID_COLUMNS]
X, y, groups = df[feature_columns], df["genre"], df["artist_id"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
print("Samples:", len(df), "Features:", len(feature_columns))


Samples: 7997 Features: 61


In [2]:
models = {
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=42)),
        ]),
        {"model__C": [0.1, 1.0, 10.0]},
    ),
    "KNN": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier()),
        ]),
        {"model__n_neighbors": [3,5,7], "model__weights": ["uniform","distance"]},
    ),
    "SVM": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, random_state=42)),
        ]),
        {"model__C": [1,10], "model__kernel": ["rbf"], "model__gamma": ["scale","auto"]},
    ),
    "Random Forest": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
        ]),
        {"model__n_estimators": [200], "model__max_depth": [None,20], "model__min_samples_leaf": [1,2]},
    ),
}


In [3]:
results, searches = [], {}

for name, (pipeline, param_grid) in models.items():
    print(f"\nTraining: {name}")
    search = GridSearchCV(
        pipeline, param_grid, scoring="f1_macro",
        cv=cv, n_jobs=-1, refit=True, return_train_score=True
    )
    search.fit(X, y, groups=groups)
    searches[name] = search
    results.append({
        "model": name,
        "best_macro_f1": search.best_score_,
        "best_params": search.best_params_,
    })
    print("Best macro F1:", search.best_score_)
    print("Best params:", search.best_params_)

results_df = pd.DataFrame(results).sort_values("best_macro_f1", ascending=False)
results_df



Training: Logistic Regression
Best macro F1: 0.42581161984029176
Best params: {'model__C': 1.0}

Training: KNN
Best macro F1: 0.38678522046461955
Best params: {'model__n_neighbors': 7, 'model__weights': 'distance'}

Training: SVM
Best macro F1: 0.4726343023443146
Best params: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}

Training: Random Forest
Best macro F1: 0.43747710393617006
Best params: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 200}


,model,best_macro_f1,best_params
2,SVM,0.472634,"{'model__C': 1, 'model__gamma': 'scale', 'mode..."
3,Random Forest,0.437477,"{'model__max_depth': None, 'model__min_samples..."
0,Logistic Regression,0.425812,{'model__C': 1.0}
1,KNN,0.386785,"{'model__n_neighbors': 7, 'model__weights': 'd..."


In [5]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=len(label_encoder.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_param_grid = {
    "n_estimators": [200],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

In [6]:
xgb_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=xgb_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X, y_encoded, groups=groups)

print("Best XGBoost Macro-F1:", xgb_search.best_score_)
print("Best parameters:", xgb_search.best_params_)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best XGBoost Macro-F1: 0.46611897777519173
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}


In [4]:
out = PROJECT_ROOT / "results" / "model_evaluation"
out.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out / "model_comparison.csv", index=False)
print("Saved:", out / "model_comparison.csv")


Saved: d:\ML-clgProject\Epochs-Music-Genre-Classifier\results\model_evaluation\model_comparison.csv
